# 원욱님 STFPM 84% vs 지도학습 97% — 차이가 어디서 오는지

한 번에 한 가지씩만 바꾼 실험 세 개를 기존 결과와 나란히 놓는다.

| 실험 | 방식 | 자르는 법 | 사진 | 분할 | 비교해서 보는 것 |
| --- | --- | --- | --- | --- | --- |
| ① | STFPM(원욱님 코드) | 고정 위치 | 전체 4,000 | 원욱님 방식(정상 5장마다 1장 테스트) | 84% 재현 |
| ② | STFPM | 고정 위치 | 전체 4,000 | 시간 블록 5-fold | ①→② **분할** 효과 |
| ③ | 지도학습 ResNet18 | 고정 위치 | 전체 4,000 | 시간 블록 5-fold | ②→③ **방식** 효과 |
| 기존 | 지도학습 ResNet18 | 이동+마스킹 | 자동 선별 3,425 | 시간 블록 5-fold | ③→기존 **전처리·선별** 효과 |

**준비:** Drive `산공종설/`에 두 파일을 올린다.
- `underfill_originals_4000.zip`: 원본 사진 4,000장(약 1.1GB)
- `stfpm_compare_bundle.zip`: 원욱님 코드(수정 없음), 분할표 2개, 실행 스크립트

위에서부터 차례로 실행한다. 런타임은 **L4**.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > L4"

## 2. Drive 연결
결과는 Drive `산공종설/results/`에 바로 저장한다. 끊겨도 끝난 부분은 남는다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS_DIR = f"{DRIVE_DIR}/results"
for f in ("underfill_originals_4000.zip", "stfpm_compare_bundle.zip"):
    assert os.path.exists(f"{DRIVE_DIR}/{f}"), f"Drive 산공종설/ 에 {f} 없음"
print("ok")

## 3. 복사 → 압축 해제
Drive에서 파일을 하나씩 읽으면 느리니까, zip을 Colab 디스크로 복사해서 푼다.
- `/content/orig`: 원본 사진. 마지막 줄 숫자가 4000이어야 한다.
- `/content/bundle`: 원욱님 코드(`project_STFPM/`), 분할표, 실행 스크립트

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/stfpm_compare_bundle.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/stfpm_compare_bundle.zip -d /content/bundle
!ls /content/bundle
!find /content/orig -name "*.jpg" | wc -l

## 4. anomalib 설치
원욱님 코드는 `anomalib`(이상탐지 라이브러리)을 쓴다. 원욱님 환경과 같은 2.6.2 버전을 깐다.
- 설치하다가 "세션 다시 시작" 안내가 뜨면 다시 시작한다. 그다음 **1~3번 셀을 다시 실행**하고 이 셀은 건너뛴다. 설치한 패키지는 남아 있다.
- 마지막 줄에 `cuda True`가 나와야 한다.

In [ ]:
!pip install -q "anomalib==2.6.2" timm
import importlib.metadata as md, torch
print({p: md.version(p) for p in ("anomalib", "torch", "torchvision", "lightning")}, "| cuda", torch.cuda.is_available())

## 5. ① STFPM, 원욱님 방식 분할 (84% 재현 확인)
정상 2,000장을 프레임 순서로 세워서 5장마다 1장(400장)을 테스트로 뺀다. 나머지 1,600장으로 학습하고, 언더필 2,000장은 전부 테스트로 쓴다. 원욱님 설명과 같은 구성이다.

`stfpm_timeblock.py`가 하는 일:
1. 분할표대로 `train/good`, `test/good`, `test/defect` 폴더를 바로가기(링크)로 만든다. 사진을 복사하지 않아서 빠르다.
2. 원욱님 `src/train.py`를 **그대로** 실행한다. 설정 파일은 경로만 바꾼 사본을 쓴다.
3. 원욱님 판정기(`Printer3DDefectPredictor`)로 테스트 사진마다 점수를 매겨 `scores.csv`에 저장한다.

한 번 학습에 몇 분 걸린다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/orig --split-csv /content/bundle/stride5_split.csv --out "{RESULTS_DIR}/stfpm_resnet18_stride5"

## 6. ② STFPM, 시간 블록 5-fold
①과 모델·전처리는 똑같고 **분할만** 바꾼다. fold마다 학습 구간의 정상 사진으로만 학습하고, 검증 구간의 정상 400 + 언더필 400으로 평가한다. 검증 사진과 30프레임 이내인 학습 사진은 뺀다.

①보다 점수가 떨어지면 그만큼이 "이웃 프레임을 외워서 생긴 이득"이다. fold가 5개라 ①의 약 5배 걸린다. 도중에 끊기면 이 셀을 다시 실행하면 된다. 끝난 fold는 건너뛴다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/orig --split-csv /content/bundle/folds_all4000.csv --out "{RESULTS_DIR}/stfpm_resnet18_timeblock_all4000"

## 7. ③ 지도학습 ResNet18, 같은 입력·같은 분할
②와 입력(원본 → 원욱님과 같은 고정 위치 자르기 → 512)과 분할이 같다. **방식만** 지도학습으로 바꾼다.
- `--input-column source --fixed-crop 0 250 1000 1250`: 원본 사진을 원욱님 코드와 같은 위치(세로 0~1000, 가로 250~1250)에서 자른다.
- `--img-size 512`: 원욱님 설정과 같은 크기다.

②와 ③의 차이가 "라벨을 쓰느냐 마느냐"의 효과다.

In [ ]:
OUT3 = f"{RESULTS_DIR}/resnet18_fixedcrop512_all4000_fast_seeds0"
if os.path.exists(f"{OUT3}/summary.csv"):
    print("이미 끝남:", OUT3)
else:
    !python /content/bundle/train_pilot_classifier.py --folds-csv /content/bundle/folds_all4000.csv --pilot-root /content/orig --input-column source --fixed-crop 0 250 1000 1250 --img-size 512 --arch resnet18 --seeds 0 --fast --output "{OUT3}"

## 8. 비교표
**읽는 법**
- `accuracy`: STFPM은 원욱님 방식의 판정 기준(정상 학습 점수 상위 1%)을 쓰고, 지도학습은 확률 0.5를 기준으로 쓴다.
- `underfill_recall`: 언더필을 잡은 비율 / `normal_recall`: 정상을 정상으로 맞힌 비율
- `auroc`: 판정 기준과 상관없이 두 클래스를 얼마나 잘 가르는지. 0.5는 찍기, 1은 완벽이다. STFPM은 fold별 AUROC의 평균이다(fold마다 모델이 달라 점수 눈금이 다르기 때문).
- 기존 줄은 사진 수가 다르다(선별 3,425장). 그래서 ③과 비교할 때 "전처리와 선별을 합친 효과"로만 봐야 한다.

이 셀 출력을 그대로 Claude에게 붙여 주면 된다.

In [ ]:
import json, pandas as pd
rows = []
def stfpm(name, path):
    s = json.load(open(f"{path}/summary.json"))
    p = s["pooled"]
    rows.append({"run": name, "accuracy": p["accuracy"], "underfill_recall": p["underfill_recall"],
                 "normal_recall": p["normal_recall"], "auroc": s["fold_mean_auroc"], "n_test": p["n"]})
def supervised(name, path):
    s = pd.read_csv(f"{path}/summary.csv", encoding="utf-8-sig")
    for _, r in s.iterrows():
        rows.append({"run": f"{name} ({r['mode']})", "accuracy": r["pooled_mean_accuracy"],
                     "underfill_recall": r["pooled_mean_underfill_recall"], "normal_recall": r["pooled_mean_normal_recall"],
                     "auroc": r["pooled_mean_auroc"], "n_test": None})
for name, path, fn in (("① STFPM 원욱님분할", f"{RESULTS_DIR}/stfpm_resnet18_stride5", stfpm),
                       ("② STFPM 시간블록", f"{RESULTS_DIR}/stfpm_resnet18_timeblock_all4000", stfpm),
                       ("③ 지도 고정크롭", OUT3, supervised),
                       ("기존 지도 마스킹·선별", f"{RESULTS_DIR}/resnet18_keep3425_masked_500_fast_seeds0", supervised)):
    try:
        fn(name, path)
    except FileNotFoundError:
        print("아직 없음:", name)
print(pd.DataFrame(rows).round(3).to_string(index=False))
for name, sub in (("①", "stfpm_resnet18_stride5"), ("②", "stfpm_resnet18_timeblock_all4000")):
    p = f"{RESULTS_DIR}/{sub}/summary.json"
    if os.path.exists(p):
        pf = json.load(open(p))["per_fold"]
        print(name, "fold별 acc/언더필/정상/auroc:",
              {k: [round(v[m], 3) for m in ("accuracy", "underfill_recall", "normal_recall", "auroc")] for k, v in pf.items()})

## 9. 런타임 반납
Colab 사용량은 GPU 런타임에 연결된 시간만큼 빠진다. 5~8번을 줄 세워 두고 자리를 비울 거면, 이 셀도 같이 실행 대기시켜 둔다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()   # Drive에 덜 올라간 파일을 마저 올리고 연결 해제
runtime.unassign()          # 런타임 반납